In [57]:
import csv
import matplotlib.pyplot as plt
import numpy as np
import torch
import transformers
import pandas as pd
import re
import ast
import warnings
import evaluate
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer, TrainerCallback
from datasets import Dataset
import logging

warnings.filterwarnings("ignore", category=RuntimeWarning)
logging.getLogger("transformers.tokenization_utils_base").setLevel(logging.ERROR)

Random seed, model and df set

In [58]:
np.random.seed(42)
model_name = "Jean-Baptiste/camembert-ner"
df = pd.read_csv("../src/data/raw/train_v2.csv", sep=',')
df = df.drop(columns=['is_comic', 'comic_name', 'tokens'])
df.head()

,Unnamed: 0,video_name,is_name
0,0,Le Barbecue Disney - La chanson de Frédéric Fr...,"[0, 0, 0, 0, 0, 0, 0, 1, 1]"
1,1,Le Roi et l'Oiseau - La Chronique de Christine...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]"
2,2,L'amour du lac - La chronique d'Hippolyte Gira...,"[0, 0, 0, 0, 0, 0, 0, 0, 1, 1]"
3,3,La fille de la piscine de Léa Tourret - La chr...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]"
4,4,"""Le soleil va moins faire son malin quand Jean...","[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ..."


Prendre les phrases et les labels

In [59]:
def make_labelled_sentences(df):
    sentences = []
    labels = []
    df['is_name'] = df['is_name'].apply(lambda x: ast.literal_eval(x))
    for i, row in df.iterrows():
        sentence = row['video_name']
        label = row['is_name']
        sentence = [word for word in re.split(r"[ ']", sentence)]
        label = [label for label in label]
            
        sentences.append(sentence)
        labels.append(label)

    return sentences, labels

In [60]:
sentences, labels = make_labelled_sentences(df)

Clean dataset

In [61]:
def delete_line_with_desequal_length(sentences, labels):
    for i in reversed(range(len(sentences))):
        if len(sentences[i]) != len(labels[i]):
            del sentences[i]
            del labels[i]
    return sentences, labels

In [62]:
sentences, labels = delete_line_with_desequal_length(sentences, labels)

Set global dataset on clean dataset

In [63]:
sentences_training, sentences_test, labels_training, labels_test = train_test_split(
    sentences,
    labels,
    test_size=0.2,
    random_state=42,
)

In [64]:
sentences_train, sentences_dev, labels_train, labels_dev = train_test_split(
    sentences_training,
    labels_training,
    test_size=0.2,
    random_state=42,
)

Tokenizer and model

In [65]:
tokenizer = AutoTokenizer.from_pretrained("Jean-Baptiste/camembert-ner", add_prefix_space=True)
model = AutoModelForTokenClassification.from_pretrained("Jean-Baptiste/camembert-ner", num_labels=2, ignore_mismatched_sizes=True)

Some weights of CamembertForTokenClassification were not initialized from the model checkpoint at Jean-Baptiste/camembert-ner and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([5]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([5, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [66]:
def tokenize_and_align_labels(sentences, ner_tags):
    tokenized_inputs = tokenizer(
        sentences,
        truncation=True,
        is_split_into_words=True,
    )
    labels = []
    for i, label in enumerate(ner_tags):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [67]:
tokenized_train = tokenize_and_align_labels(sentences_train, labels_train)
tokenized_test = tokenize_and_align_labels(sentences_test, labels_test)

In [68]:
dataset_train = Dataset.from_dict(tokenized_train)
dataset_test = Dataset.from_dict(tokenized_test)

In [69]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [70]:
seqeval = evaluate.load("seqeval")

labels = [0, 1]
label_list = ["0", "1"]

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [71]:
"""
for name, param in model.base_model.named_parameters():
  param.requires_grad = False

for name, param in model.base_model.named_parameters():
    if (
        any(layer_name in name for layer_name in ["layer.0", "layer.1", "layer.2", "layer.3", "layer.4"])
        and any(layer_type in name for layer_type in ["weight", "bias"])
        and "attention" not in name
    ):
        param.requires_grad = True
"""

'\nfor name, param in model.base_model.named_parameters():\n  param.requires_grad = False\n\nfor name, param in model.base_model.named_parameters():\n    if (\n        any(layer_name in name for layer_name in ["layer.0", "layer.1", "layer.2", "layer.3", "layer.4"])\n        and any(layer_type in name for layer_type in ["weight", "bias"])\n        and "attention" not in name\n    ):\n        param.requires_grad = True\n'

In [ ]:
training_args = TrainingArguments(
    output_dir="my_awesome_wnut_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[StoreMetricsCallback()]
)

trainer.train()

Epoch,Training Loss,Validation Loss


C:\Users\erwan\miniconda3\envs\NLP\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: 1 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\erwan\miniconda3\envs\NLP\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\erwan\miniconda3\envs\NLP\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\erwan\miniconda3\envs\NLP\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
Checkpoint desti

In [ ]:
trainer.evaluate()

In [ ]:
trainer.save_model("model_finetuned")